# A1: NLP Pipeline and Linguistic Analysis

Apply and analyse the basic stages of NLP on a text corpus: tokenization, stop-word removal, stemming / lemmatization, POS tagging, morphological components (root, suffix), ambiguity (lexical / syntactic), and compare NLTK with spaCy.

**Dataset:** a short sample text, and 200 IMDB movie reviews for the library comparison

In [2]:
# nltk  : tokenizer, stop words, stemmer, lemmatizer, POS tagger, WordNet
# spacy : the second library to compare with (en_core_web_sm = small English model)
!pip install -q nltk spacy
!python -m spacy download en_core_web_sm

In [3]:
import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng", "universal_tagset"]:
    nltk.download(pkg, quiet=True)

import spacy
import pandas as pd
nlp = spacy.load("en_core_web_sm")

## Sample text

In [4]:
text = "The students were studying NLP in Pune. They visited the bank of the river after the classes were finished. I saw the man with the telescope."
text

'The students were studying NLP in Pune. They visited the bank of the river after the classes were finished. I saw the man with the telescope.'

## 1. Tokenization

In [5]:
from nltk.tokenize import word_tokenize, sent_tokenize

sentences = sent_tokenize(text)
sentences

['The students were studying NLP in Pune.',
 'They visited the bank of the river after the classes were finished.',
 'I saw the man with the telescope.']

In [6]:
tokens = word_tokenize(text)
print(len(tokens), "tokens")
print(tokens)

29 tokens
['The', 'students', 'were', 'studying', 'NLP', 'in', 'Pune', '.', 'They', 'visited', 'the', 'bank', 'of', 'the', 'river', 'after', 'the', 'classes', 'were', 'finished', '.', 'I', 'saw', 'the', 'man', 'with', 'the', 'telescope', '.']


In [7]:
doc = nlp(text)
spacy_tokens = [t.text for t in doc]
print(len(spacy_tokens), "tokens")
print(spacy_tokens)

29 tokens
['The', 'students', 'were', 'studying', 'NLP', 'in', 'Pune', '.', 'They', 'visited', 'the', 'bank', 'of', 'the', 'river', 'after', 'the', 'classes', 'were', 'finished', '.', 'I', 'saw', 'the', 'man', 'with', 'the', 'telescope', '.']


## 2. Stop-word removal

In [8]:
from nltk.corpus import stopwords
swords = stopwords.words("english")

words = [t.lower() for t in tokens if t.isalpha()]          # keep only words, lowercase
no_stop = [w for w in words if w not in swords]
print(no_stop)

['students', 'studying', 'nlp', 'pune', 'visited', 'bank', 'river', 'classes', 'finished', 'saw', 'man', 'telescope']


In [9]:
# spaCy marks stop words on each token
print([t.text for t in doc if t.is_alpha and not t.is_stop])

['students', 'studying', 'NLP', 'Pune', 'visited', 'bank', 'river', 'classes', 'finished', 'saw', 'man', 'telescope']


## 3. Stemming and lemmatization

In [10]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
ps = PorterStemmer()
wnl = WordNetLemmatizer()

pd.DataFrame({"word": no_stop,
              "stem (Porter)": [ps.stem(w) for w in no_stop],
              "lemma (WordNet, as verb)": [wnl.lemmatize(w, pos="v") for w in no_stop]})

,word,stem (Porter),"lemma (WordNet, as verb)"
0,students,student,students
1,studying,studi,study
2,nlp,nlp,nlp
3,pune,pune,pune
4,visited,visit,visit
5,bank,bank,bank
6,river,river,river
7,classes,class,class
8,finished,finish,finish
9,saw,saw,saw


In [11]:
# spaCy lemmatizes using the POS of each word
pd.DataFrame([(t.text, t.lemma_) for t in doc if t.is_alpha], columns=["word", "spaCy lemma"])

,word,spaCy lemma
0,The,the
1,students,student
2,were,be
3,studying,study
4,NLP,NLP
5,in,in
6,Pune,Pune
7,They,they
8,visited,visit
9,the,the


## 4. POS tagging

In [12]:
nltk_pos = nltk.pos_tag(tokens)                         # Penn Treebank tags (NN, VBD, ...)
nltk_uni = nltk.pos_tag(tokens, tagset="universal")     # universal tags (NOUN, VERB, ...)
spacy_pos = [(t.text, t.tag_, t.pos_) for t in doc]

pd.DataFrame({"token": tokens,
              "NLTK (Penn)": [p for _, p in nltk_pos],
              "NLTK (universal)": [p for _, p in nltk_uni],
              "spaCy (Penn)": [t.tag_ for t in doc] if len(doc) == len(tokens) else None,
              "spaCy (universal)": [t.pos_ for t in doc] if len(doc) == len(tokens) else None})

,token,NLTK (Penn),NLTK (universal),spaCy (Penn),spaCy (universal)
0,The,DT,DET,DT,DET
1,students,NNS,NOUN,NNS,NOUN
2,were,VBD,VERB,VBD,AUX
3,studying,VBG,VERB,VBG,VERB
4,NLP,NNP,NOUN,NNP,PROPN
5,in,IN,ADP,IN,ADP
6,Pune,NNP,NOUN,NNP,PROPN
7,.,.,.,.,PUNCT
8,They,PRP,PRON,PRP,PRON
9,visited,VBD,VERB,VBD,VERB


## 5. Morphological components: root and suffix

In [13]:
def root_and_suffix(word):
    # candidate roots: lemma as verb, lemma as noun, Porter stem; keep the shortest one the word starts with
    candidates = [wnl.lemmatize(word, "v"), wnl.lemmatize(word, "n"), ps.stem(word)]
    roots = [c for c in candidates if word.startswith(c)]
    root = min(roots, key=len) if roots else word
    return root, word[len(root):]

for w in ["studying", "students", "visited", "classes", "finished", "happiness", "reading"]:
    print(f"{w:<10} -> root: {root_and_suffix(w)[0]:<8} suffix: {root_and_suffix(w)[1]}")

studying   -> root: study    suffix: ing
students   -> root: student  suffix: s
visited    -> root: visit    suffix: ed
classes    -> root: class    suffix: es
finished   -> root: finish   suffix: ed
happiness  -> root: happi    suffix: ness
reading    -> root: read     suffix: ing


In [14]:
# spaCy gives grammatical features of each word
[(t.text, t.lemma_, str(t.morph)) for t in doc if t.pos_ in ("NOUN", "VERB", "AUX")]

[('students', 'student', 'Number=Plur'),
 ('were', 'be', 'Mood=Ind|Tense=Past|VerbForm=Fin'),
 ('studying', 'study', 'Aspect=Prog|Tense=Pres|VerbForm=Part'),
 ('visited', 'visit', 'Tense=Past|VerbForm=Fin'),
 ('bank', 'bank', 'Number=Sing'),
 ('river', 'river', 'Number=Sing'),
 ('classes', 'class', 'Number=Plur'),
 ('were', 'be', 'Mood=Ind|Tense=Past|VerbForm=Fin'),
 ('finished', 'finish', 'Aspect=Perf|Tense=Past|VerbForm=Part'),
 ('saw', 'see', 'Tense=Past|VerbForm=Fin'),
 ('man', 'man', 'Number=Sing'),
 ('telescope', 'telescope', 'Number=Sing')]

## 6. Ambiguity

**Lexical ambiguity:** one word, several meanings. WordNet lists the senses of a word.

In [15]:
from nltk.corpus import wordnet as wn

for s in wn.synsets("bank")[:6]:
    print(s.name(), "-", s.definition())
print("\nnumber of senses of 'bank':", len(wn.synsets("bank")))

bank.n.01 - sloping land (especially the slope beside a body of water)
depository_financial_institution.n.01 - a financial institution that accepts deposits and channels the money into lending activities
bank.n.03 - a long ridge or pile
bank.n.04 - an arrangement of similar objects in a row or in tiers
bank.n.05 - a supply or stock held in reserve for future use (especially in emergencies)
bank.n.06 - the funds held by a gambling house or the dealer in some gambling games

number of senses of 'bank': 18


**Syntactic ambiguity:** one sentence, several structures. "I saw the man with the telescope": who has the telescope? A small grammar gives both parse trees.

In [16]:
grammar = nltk.CFG.fromstring("""
S  -> NP VP
NP -> 'I' | Det N | Det N PP
VP -> V NP | VP PP
PP -> P NP
Det -> 'the'
N  -> 'man' | 'telescope'
V  -> 'saw'
P  -> 'with'
""")
parser = nltk.ChartParser(grammar)
trees = list(parser.parse("I saw the man with the telescope".split()))
print(len(trees), "parse trees")
for t in trees:
    t.pretty_print()

2 parse trees
     S                                    
  ___|___________                          
 |               VP                       
 |        _______|________                 
 |       VP               PP              
 |    ___|___         ____|___             
 |   |       NP      |        NP          
 |   |    ___|___    |     ___|______      
 NP  V  Det      N   P   Det         N    
 |   |   |       |   |    |          |     
 I  saw the     man with the     telescope

     S                                
  ___|_______                          
 |           VP                       
 |    _______|___                      
 |   |           NP                   
 |   |    _______|____                 
 |   |   |   |        PP              
 |   |   |   |    ____|___             
 |   |   |   |   |        NP          
 |   |   |   |   |     ___|______      
 NP  V  Det  N   P   Det         N    
 |   |   |   |   |    |          |     
 I  saw the man with the     tele

**Classify the type** of ambiguity for a few sentences, with the evidence.

In [17]:
examples = pd.DataFrame([
    ("They visited the bank of the river", "bank", "lexical: bank = river side or money bank"),
    ("I saw the man with the telescope", None, "syntactic: who has the telescope? (2 parse trees above)"),
    ("She is looking for a match", "match", "lexical: match = game or matchstick"),
    ("Old men and women were invited", None, "syntactic: are the women old too?")],
    columns=["sentence", "ambiguous word", "type"])
examples["senses of the word in WordNet"] = [len(wn.synsets(w)) if isinstance(w, str) else "-" for w in examples["ambiguous word"]]
examples

,sentence,ambiguous word,type,senses of the word in WordNet
0,They visited the bank of the river,bank,lexical: bank = river side or money bank,18
1,I saw the man with the telescope,NaN,syntactic: who has the telescope? (2 parse tre...,-
2,She is looking for a match,match,lexical: match = game or matchstick,19
3,Old men and women were invited,NaN,syntactic: are the women old too?,-


## 7. Compare NLTK and spaCy on 200 IMDB reviews

In [18]:
imdb = pd.read_csv("https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv")
reviews = imdb["review"].str.replace("<br />", " ").head(200)
reviews.head(3)

0    One of the other reviewers has mentioned that ...
1    A wonderful little production.   The filming t...
2    I thought this was a wonderful way to spend ti...
Name: review, dtype: str

In [19]:
import time

t0 = time.time()
nltk_tok = [word_tokenize(r) for r in reviews]
nltk_pos = [nltk.pos_tag(t, tagset="universal") for t in nltk_tok]
nltk_time = time.time() - t0

t0 = time.time()
docs = list(nlp.pipe(reviews))
spacy_time = time.time() - t0

pd.DataFrame({"NLTK": [sum(len(t) for t in nltk_tok), round(nltk_time, 1)],
              "spaCy": [sum(len(d) for d in docs), round(spacy_time, 1)]},
             index=["total tokens", "seconds (tokenize + POS)"])

,NLTK,spaCy
total tokens,54174.0,55450.0
seconds (tokenize + POS),2.0,7.4


In [20]:
# spaCy and NLTK name some universal tags differently; map spaCy's names to NLTK's before comparing
to_nltk = {"AUX": "VERB", "PROPN": "NOUN", "CCONJ": "CONJ", "SCONJ": "ADP", "PART": "PRT", "PUNCT": ".", "SYM": "X", "INTJ": "X", "SPACE": "X"}
spacy_tag = lambda t: to_nltk.get(t.pos_, t.pos_)

same = agree = 0
for toks, tags, d in zip(nltk_tok, nltk_pos, docs):
    if toks == [t.text for t in d]:                      # compare only where both split the text the same way
        same += 1
        agree += sum(a[1] == spacy_tag(b) for a, b in zip(tags, d)) / len(toks)
print(f"same tokens in {same} of 200 reviews; average POS agreement there: {agree / same:.1%}")

same tokens in 18 of 200 reviews; average POS agreement there: 90.8%


In [21]:
# where they disagree, on one review
toks, tags, d = next((a, b, c) for a, b, c in zip(nltk_tok, nltk_pos, docs) if a == [t.text for t in c])
diff = [(w, a, spacy_tag(b)) for (w, a), b in zip(tags, d) if a != spacy_tag(b)]
pd.DataFrame(diff, columns=["word", "NLTK", "spaCy"]).head(10)

,word,NLTK,spaCy
0,george,ADJ,NOUN
1,not,ADV,PRT
2,not,ADV,PRT
3,to,PRT,ADP
4,everybody,NOUN,PRON


## Conclusion
- Both libraries tokenize, remove stop words, lemmatize and tag; spaCy does it in one call, NLTK step by step.
- Stemming cuts suffixes by rules (studi, happi), lemmatization returns real words (study).
- Lexical ambiguity shows up as many WordNet senses; syntactic ambiguity as more than one parse tree.
- The tokenizers split contractions and punctuation differently, so token counts differ; after mapping tag names the taggers mostly agree.
- WordNet's lemmatizer needs the right POS ("saw" stays "saw"), spaCy picks the POS itself ("saw" -> "see").